# Polars Lazy API: план выполнения запрос к Silver-слою Delta Lake

В этом Jupyter Notebook мы смотрим на то, как Polars планирует выполнение запроса
к таблице Silver перед тем как реально читать данные с диска.

Данный notebook обращается к Silver-таблице Delta Lake и строит запрос
на вычисление средней задержки прилёта по сезонам для рейсов из аэропорта ATL.


Запрос строится в режиме Lazy API — это означает что данные не загружаются
в память сразу. Сначала Polars составляет план выполнения, оптимизирует его,
и только после вызова `.collect()` фактически читает данные с диска.

Чтобы увидеть этот план до выполнения запроса, используется метод `.explain()`.
Из его вывода видно какие именно колонки и строки Polars собирается читать —
и как он сокращает объём данных ещё до их загрузки в память.

In [2]:
import polars as pl
import os

SILVER_PATH = os.path.join("..", "delta", "silver")

query = (
    pl.scan_delta(SILVER_PATH)
    .filter(pl.col("Origin") == "ATL")
    .select(["FlightDate", "Origin", "Dest", "ArrDelay", "season"])
    .group_by("season")
    .agg(pl.col("ArrDelay").mean().alias("avg_delay"))
)

print(query.explain())

AGGREGATE[maintain_order: false]
  [col("ArrDelay").mean().alias("avg_delay")] BY [col("season")]
  FROM
  simple π 2/2 ["ArrDelay", "season"]
    Parquet SCAN [C:/projects/3_lab/delta/silver/FlightDate=2024-01-31/part-00000-2c9d9c8d-4634-4b73-9149-29e411908bdb-c000.zstd.parquet, ... 30 other sources]
    PROJECT 3/16 COLUMNS
    SELECTION: [(col("Origin")) == ("ATL")]
